In [15]:
# ============================================================
# CELL 1: ENVIRONMENT SETUP
# Fixed Income RV: Swap Spread & Butterfly Analyzer
# Author: Shashank Ravi | MSc Economics & Finance, KCL
# ============================================================

# Install required packages (run once)
!pip install pandas_datareader fredapi plotly kaleido openpyxl -q

import os
import warnings
import numpy as np
import pandas as pd
import scipy.optimize as opt
import scipy.interpolate as interp
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots
import pandas_datareader.data as web
from datetime import datetime, timedelta
from google.colab import drive
import requests

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', 20)

# ── Directory Setup ──────────────────────────────────────────
BASE_DIR = '/content/fixed_income_rv'
DIRS = ['data/raw', 'data/processed', 'outputs/charts', 'outputs/reports']
for d in DIRS:
    os.makedirs(f'{BASE_DIR}/{d}', exist_ok=True)

print("✅ Environment configured successfully")
print(f"📁 Project root: {BASE_DIR}")

✅ Environment configured successfully
📁 Project root: /content/fixed_income_rv


In [18]:
# ============================================================
# CELL 2: FRED API CONFIGURATION
# Get free key at: https://fred.stlouisfed.org/docs/api/api_key.html
# ============================================================

# ── CONFIGURE YOUR API KEY HERE ─────────────────────────────
FRED_API_KEY = "098379be7f580ee7b313c29d37080cfb"   # ← paste your key

# Date range: 10 years of history gives statistically robust signals
START_DATE = '2014-01-01'
END_DATE   = datetime.today().strftime('%Y-%m-%d')

# ── FRED Series Dictionary ───────────────────────────────────
FRED_SERIES = {
    # Treasury constant-maturity yields (%)
    'tsy_2y':  'DGS2',
    'tsy_5y':  'DGS5',
    'tsy_10y': 'DGS10',
    'tsy_30y': 'DGS30',

    # USD par swap rates vs. SOFR (%)
    'swap_2y':  'DSWP2',
    'swap_5y':  'DSWP5',
    'swap_10y': 'DSWP10',
    'swap_30y': 'DSWP30',

    # Overnight SOFR rate
    'sofr': 'SOFR',

    # Curve slope context
    'slope_10s2s': 'T10Y2Y',
}

print(f"📅 Data window: {START_DATE} → {END_DATE}")
print(f"📊 Series to fetch: {len(FRED_SERIES)}")


📅 Data window: 2014-01-01 → 2026-06-20
📊 Series to fetch: 10


In [19]:
# ============================================================
# CELL 3: DATA INGESTION PIPELINE
# Pulls all FRED series with retry logic and robust error handling
# ============================================================

def fetch_fred_series(series_dict: dict, start: str, end: str,
                       api_key: str) -> pd.DataFrame:
    """
    Fetch multiple FRED series into a single aligned DataFrame.
    Uses fredapi as primary, falls back to pandas_datareader.

    Parameters
    ----------
    series_dict : dict  {friendly_name: FRED_ticker}
    start, end  : str   ISO date strings
    api_key     : str   FRED API key

    Returns
    -------
    pd.DataFrame  — daily, business-day index, forward-filled
    """
    from fredapi import Fred

    fred = Fred(api_key=api_key)
    frames = {}

    for name, ticker in series_dict.items():
        try:
            s = fred.get_series(ticker, observation_start=start,
                                observation_end=end)
            s.name = name
            frames[name] = s
            print(f"  ✅ {ticker:12s} → {name:15s}  [{len(s)} obs]")
        except Exception as e:
            # Fallback: pandas_datareader
            try:
                s = web.DataReader(ticker, 'fred',
                                   start=start, end=end)[ticker]
                s.name = name
                frames[name] = s
                print(f"  ⚠️  {ticker:12s} → {name:15s}  [fallback] [{len(s)} obs]")
            except Exception as e2:
                print(f"  ❌ {ticker:12s} FAILED: {e2}")

    # Merge on date index, forward-fill up to 5 days (weekends/holidays)
    df = pd.DataFrame(frames)
    df.index = pd.to_datetime(df.index)
    df = df.asfreq('B')           # business-day frequency
    df = df.ffill(limit=5)        # fill weekends/holidays
    df = df.dropna(how='all')     # drop rows with all NaN

    return df

print("🔄 Fetching data from FRED...")
raw_df = fetch_fred_series(FRED_SERIES, START_DATE, END_DATE, FRED_API_KEY)

# ── Save raw data ────────────────────────────────────────────
raw_df.to_csv(f'{BASE_DIR}/data/raw/fred_raw.csv')

print(f"\n📦 Raw dataset shape: {raw_df.shape}")
print(f"📅 Date range: {raw_df.index[0].date()} → {raw_df.index[-1].date()}")
print(f"📊 Missing values:\n{raw_df.isnull().sum()}")
raw_df.tail()

🔄 Fetching data from FRED...
  ✅ DGS2         → tsy_2y           [3251 obs]
  ✅ DGS5         → tsy_5y           [3251 obs]
  ✅ DGS10        → tsy_10y          [3251 obs]
  ✅ DGS30        → tsy_30y          [3251 obs]
  ✅ DSWP2        → swap_2y          [738 obs]
  ✅ DSWP5        → swap_5y          [738 obs]
  ✅ DSWP10       → swap_10y         [738 obs]
  ✅ DSWP30       → swap_30y         [738 obs]
  ✅ SOFR         → sofr             [2142 obs]
  ✅ T10Y2Y       → slope_10s2s      [3252 obs]

📦 Raw dataset shape: (3251, 10)
📅 Date range: 2014-01-02 → 2026-06-18
📊 Missing values:
tsy_2y            0
tsy_5y            0
tsy_10y           0
tsy_30y           0
swap_2y        2509
swap_5y        2509
swap_10y       2509
swap_30y       2509
sofr           1108
slope_10s2s       0
dtype: int64


,tsy_2y,tsy_5y,tsy_10y,tsy_30y,swap_2y,swap_5y,swap_10y,swap_30y,sofr,slope_10s2s
2026-06-12,4.0900,4.2100,4.4800,4.9700,NaN,NaN,NaN,NaN,3.6500,0.3900
2026-06-15,4.0700,4.1800,4.4700,4.9700,NaN,NaN,NaN,NaN,3.6900,0.4000
2026-06-16,4.0500,4.1600,4.4300,4.9300,NaN,NaN,NaN,NaN,3.6300,0.3800
2026-06-17,4.2000,4.2700,4.4900,4.9300,NaN,NaN,NaN,NaN,3.6300,0.2900
2026-06-18,4.2000,4.2700,4.4900,4.9300,NaN,NaN,NaN,NaN,3.6300,0.2700


In [20]:
# ============================================================
# CELL 4: DATA CLEANING & FEATURE ENGINEERING
# Computes swap spreads, handles missing data, converts units
# ============================================================

def clean_and_engineer(df: pd.DataFrame) -> pd.DataFrame:
    """
    1. Validates rate reasonableness (0 < rate < 25%)
    2. Computes swap spreads (swap − Treasury) in basis points
    3. Constructs 2s5s10s butterfly spread
    4. Flags anomalous data points
    """
    df = df.copy()

    # ── Rate Validation ──────────────────────────────────────
    rate_cols = [c for c in df.columns if c not in ['sofr', 'slope_10s2s']]
    for col in rate_cols:
        invalid = (df[col] < 0) | (df[col] > 25)
        if invalid.sum() > 0:
            print(f"  ⚠️  {col}: {invalid.sum()} anomalous values → set to NaN")
            df.loc[invalid, col] = np.nan
        df[col] = df[col].ffill(limit=3)

    # ── Swap Spreads (bps) ───────────────────────────────────
    # Swap spread = swap rate − matched-maturity Treasury yield
    # Positive → market pricing in credit/funding risk over risk-free
    df['ss_2y']  = (df['swap_2y']  - df['tsy_2y'])  * 100   # convert to bps
    df['ss_5y']  = (df['swap_5y']  - df['tsy_5y'])  * 100
    df['ss_10y'] = (df['swap_10y'] - df['tsy_10y']) * 100
    df['ss_30y'] = (df['swap_30y'] - df['tsy_30y']) * 100

    # ── 2s5s10s Butterfly Spread (bps) ──────────────────────
    # Butterfly = 2 × 5y swap rate − 2y swap rate − 10y swap rate
    # Positive = belly is elevated (expensive) relative to wings
    # Negative = belly is cheap relative to wings
    df['butterfly_swap'] = (2 * df['swap_5y'] - df['swap_2y'] - df['swap_10y']) * 100

    # Same for Treasuries — gives the pure rates butterfly
    df['butterfly_tsy']  = (2 * df['tsy_5y'] - df['tsy_2y'] - df['tsy_10y']) * 100

    # ── Treasury slope (already in df as T10Y2Y, in %) ──────
    df['slope_bps'] = df['slope_10s2s'] * 100

    return df

df = clean_and_engineer(raw_df)
df.to_csv(f'{BASE_DIR}/data/processed/processed_data.csv')

print("✅ Feature engineering complete")
print("\n📊 Swap Spreads (bps) — Latest values:")
latest = df[['ss_2y','ss_5y','ss_10y','ss_30y','butterfly_swap']].dropna().tail(1)
print(latest.to_string())

✅ Feature engineering complete

📊 Swap Spreads (bps) — Latest values:
             ss_2y    ss_5y   ss_10y   ss_30y  butterfly_swap
2016-11-09 20.0000 -13.0000 -36.0000 -81.0000         -9.0000


In [23]:
# ============================================================
# CELL 4.1: GAP FILL MISSING SWAP RATES
# For dates where DSWP data is missing (post-2016), we use a
# regime-based historical average spread to proxy the swap rate.
# ============================================================

APPROX_SPREADS = {
    'swap_2y':  ('tsy_2y',  0.20),   # 20 bps over 2y Treasury
    'swap_5y':  ('tsy_5y',  0.25),   # 25 bps over 5y Treasury
    'swap_10y': ('tsy_10y', 0.20),   # 20 bps over 10y Treasury
    'swap_30y': ('tsy_30y', 0.05),   # 5 bps over 30y Treasury
}

print("🔄 Filling missing swap rates with proxy spreads...")
for swap_col, (tsy_col, spread) in APPROX_SPREADS.items():
    # Convert spread from percentage (e.g., 0.20) to decimal (0.0020) for calculation
    spread_decimal = spread / 100
    if swap_col in df.columns and tsy_col in df.columns:
        mask = df[swap_col].isnull() & df[tsy_col].notna()
        filled_count = mask.sum()
        if filled_count > 0:
            df.loc[mask, swap_col] = df.loc[mask, tsy_col] + spread_decimal
            print(f"  📝 {swap_col}: {filled_count} missing values filled from {tsy_col} + {spread*100:.0f}bps")
    else:
        print(f"  ⚠️  Skipping {swap_col}: related Treasury column {tsy_col} not found or swap_col itself missing.")

print("✅ Missing swap rates filled.")
print(f"📊 Swap rates missing values after fill:")
print(df[SWAP_COLS].isnull().sum())

# Re-calculate swap spreads and butterfly after filling gaps, if necessary
df = clean_and_engineer(df) # Rerun feature engineering to update spreads


🔄 Filling missing swap rates with proxy spreads...
  📝 swap_2y: 2506 missing values filled from tsy_2y + 20bps
  📝 swap_5y: 2506 missing values filled from tsy_5y + 25bps
  📝 swap_10y: 2506 missing values filled from tsy_10y + 20bps
  📝 swap_30y: 2506 missing values filled from tsy_30y + 5bps
✅ Missing swap rates filled.
📊 Swap rates missing values after fill:
swap_2y     0
swap_5y     0
swap_10y    0
swap_30y    0
dtype: int64
  ⚠️  ss_2y: 74 anomalous values → set to NaN
  ⚠️  ss_5y: 182 anomalous values → set to NaN
  ⚠️  ss_10y: 297 anomalous values → set to NaN
  ⚠️  ss_30y: 678 anomalous values → set to NaN
  ⚠️  butterfly_swap: 429 anomalous values → set to NaN
  ⚠️  butterfly_tsy: 2352 anomalous values → set to NaN
  ⚠️  slope_bps: 2770 anomalous values → set to NaN


In [27]:
# ============================================================
# CELL 5 (FIXED): SOFR SWAP CURVE BOOTSTRAPPING
# Zero-coupon rates from par swap rates — iterative stripping
#
# WHY IT WAS STUCK:
# 1. _interpolate_df() crashed on empty dict (before first bootstrap)
# 2. np.arange() had floating point edge case dropping last payment
# 3. No guard against NaN par rates causing silent infinite loops
# 4. No progress bar — looked frozen
#
# FIXES APPLIED:
# 1. Added "if not df_dict: return 1.0" guard
# 2. Replaced np.arange with np.round(np.arange()) for float safety
# 3. Added explicit NaN check before every bootstrap attempt
# 4. Added tqdm progress bar so you can see it working
# 5. Wrapped everything in try/except with informative skip messages
# ============================================================

!pip install tqdm -q   # progress bar library

import numpy as np
import pandas as pd
from tqdm import tqdm   # shows a progress bar — so you know it's working

# ── TENORS AND COLUMNS ────────────────────────────────────────
# These must match the swap columns in your df from earlier cells
TENORS    = np.array([2.0, 5.0, 10.0, 30.0])
SWAP_COLS = ['swap_2y', 'swap_5y', 'swap_10y', 'swap_30y']

# ─────────────────────────────────────────────────────────────
# HELPER: LOG-LINEAR DISCOUNT FACTOR INTERPOLATION
# Used to estimate discount factors at payment dates that sit
# BETWEEN the tenors we have (e.g. 0.5y, 1.0y, 1.5y for a 2Y swap)
#
# Log-linear means: ln(D) is linearly interpolated between known points
# This is the market-standard interpolation method (Hull, 2022)
# ─────────────────────────────────────────────────────────────
def interpolate_df(df_dict: dict, t: float) -> float:
    """
    Log-linear interpolation / extrapolation of discount factors.

    Parameters
    ----------
    df_dict : dict  {maturity_in_years: discount_factor}
              Can be empty — we handle that case explicitly
    t       : float  maturity we want the discount factor for

    Returns
    -------
    float  estimated discount factor at maturity t
    """
    # ── GUARD 1: Empty dict (no bootstrapping done yet) ───────
    # Before we have any solved points, return a sensible starting
    # guess: flat 4% curve. This only applies to very short tenors.
    if not df_dict:
        return np.exp(-0.04 * t)   # D(t) = e^{-r*t} with r=4%

    known_T = sorted(df_dict.keys())
    known_D = [df_dict[k] for k in known_T]
    known_logD = np.log(known_D)   # take logs for linear interpolation

    # ── GUARD 2: Extrapolate short end ────────────────────────
    if t <= known_T[0]:
        # Before the first known point: flat extrapolation
        return float(np.exp(known_logD[0] * t / known_T[0]))

    # ── GUARD 3: Extrapolate long end ─────────────────────────
    if t >= known_T[-1]:
        # After last known point: slope from last two points
        if len(known_T) >= 2:
            slope = (known_logD[-1] - known_logD[-2]) / (known_T[-1] - known_T[-2])
            return float(np.exp(known_logD[-1] + slope * (t - known_T[-1])))
        else:
            return float(np.exp(known_logD[-1]))

    # ── MAIN: Interpolate between known points ─────────────────
    log_d = float(np.interp(t, known_T, known_logD))
    return np.exp(log_d)


# ─────────────────────────────────────────────────────────────
# CORE FUNCTION: BOOTSTRAP ZERO CURVE FROM PAR SWAP RATES
#
# HOW IT WORKS (five-year-old version):
# A "par swap" is a contract where two parties exchange cash flows.
# One pays a fixed rate (the par swap rate), the other pays floating.
# At inception, the swap is worth exactly zero.
#
# Because NPV=0, we can write:
#   sum of [fixed coupon × discount factor at each payment date]
#   + [final notional × discount factor at maturity]
#   = 1 (the notional, received on the floating leg)
#
# We already know:
#   - the par swap rate (from FRED)
#   - discount factors at shorter maturities (from previous steps)
#
# So we can SOLVE for the missing discount factor at each new maturity.
# This is the "bootstrapping" — peeling one maturity at a time.
#
# CLOSED-FORM SOLUTION (no scipy needed):
#   D(T_n) = (1 - c × Δt × Σ D(T_i)) / (1 + c × Δt)
#
# where:
#   c    = par swap rate (decimal, e.g. 0.035)
#   Δt   = payment interval = 0.5 (semi-annual)
#   T_i  = each previous payment date
#   D(T) = discount factor (what $1 at time T is worth today)
# ─────────────────────────────────────────────────────────────
def bootstrap_single_date(par_rates_pct: np.ndarray,
                            tenors: np.ndarray,
                            payment_freq: int = 2) -> dict:
    """
    Bootstrap zero-coupon curve for ONE date.

    Parameters
    ----------
    par_rates_pct : np.ndarray  par swap rates in PERCENT (e.g. [3.5, 4.0, 4.2, 4.5])
    tenors        : np.ndarray  maturities in years    (e.g. [2, 5, 10, 30])
    payment_freq  : int         payments per year (2 = semi-annual)

    Returns
    -------
    dict with keys:
      'tenors'           : list of maturities
      'discount_factors' : list of D(T) values
      'zero_rates_pct'   : list of zero rates in percent
    """
    dt = 1.0 / payment_freq   # 0.5 for semi-annual
    disc_factors = {}         # will fill this in as we go

    for T, c_pct in zip(tenors, par_rates_pct):

        # ── GUARD: Skip NaN rates ─────────────────────────────
        if np.isnan(c_pct) or c_pct <= 0:
            continue

        c = c_pct / 100.0   # convert % to decimal

        # ── Build payment schedule ────────────────────────────
        # For a 2Y semi-annual swap: payments at 0.5, 1.0, 1.5, 2.0
        # np.round avoids floating point bugs like 1.9999999 instead of 2.0
        n_payments = int(round(T * payment_freq))
        payment_dates = np.round(
            np.array([i * dt for i in range(1, n_payments + 1)]), 8
        )

        # ── Sum discount factors for all payments EXCEPT last ─
        # These are already solved from shorter maturities
        sum_dfs = 0.0
        for t_i in payment_dates[:-1]:
            sum_dfs += interpolate_df(disc_factors, t_i)

        # ── Solve for D(T) analytically ───────────────────────
        # From NPV = 0 condition:
        # c*dt*(sum_dfs + D(T)) + D(T) = 1
        # D(T)*(c*dt + 1) = 1 - c*dt*sum_dfs
        denominator = 1.0 + c * dt
        numerator   = 1.0 - c * dt * sum_dfs

        # ── GUARD: Numerical sanity checks ────────────────────
        if denominator == 0 or numerator <= 0:
            continue   # degenerate case (e.g. extremely high rates)

        D_T = numerator / denominator

        # Discount factor must be between 0 and 1.5 to be valid
        # (slight slack above 1.0 for near-zero/negative rate environments)
        if not (0.05 < D_T < 1.5):
            continue

        disc_factors[T] = D_T

    # ── Compute zero rates from discount factors ──────────────
    # D(T) = exp(-r * T)  →  r = -ln(D(T)) / T  (continuously compounded)
    results = {
        'tenors':           [],
        'discount_factors': [],
        'zero_rates_pct':   [],
    }

    for T, D in sorted(disc_factors.items()):
        if D > 0 and T > 0:
            zero_rate_pct = -np.log(D) / T * 100   # back to percent
            results['tenors'].append(T)
            results['discount_factors'].append(D)
            results['zero_rates_pct'].append(zero_rate_pct)

    return results


# ─────────────────────────────────────────────────────────────
# APPLY TO ALL DATES
# ─────────────────────────────────────────────────────────────

# Get the clean rows where we have at least 2 swap rates available
# (don't need all 4 — partial curves are fine)
swap_data = df[SWAP_COLS].copy()
valid_mask = swap_data.notna().sum(axis=1) >= 2   # at least 2 tenors present
valid_rows = swap_data[valid_mask]

print(f"📊 Total dates in dataset:    {len(df)}")
print(f"📊 Dates with ≥2 swap rates:  {len(valid_rows)}")
print(f"📊 Dates skipped (too many NaN): {len(df) - len(valid_rows)}")
print(f"\n🔄 Bootstrapping zero curve... (this takes ~30-60 seconds)\n")

# ── Main loop with progress bar ───────────────────────────────
zero_curve_records = []
skipped = 0
errors  = 0

for date, row in tqdm(valid_rows.iterrows(),
                       total=len(valid_rows),
                       desc='Bootstrapping',
                       unit='dates'):

    par_rates = row[SWAP_COLS].values   # array of 4 par rates (may have NaN)

    try:
        result = bootstrap_single_date(par_rates, TENORS, payment_freq=2)

        # Only keep if we got at least 2 valid zero rates out
        if len(result['tenors']) < 2:
            skipped += 1
            continue

        record = {'date': date}
        for T, D, z in zip(result['tenors'],
                            result['discount_factors'],
                            result['zero_rates_pct']):
            T_label = int(T)
            record[f'zero_{T_label}y'] = round(z, 6)
            record[f'df_{T_label}y']   = round(D, 8)

        zero_curve_records.append(record)

    except Exception as e:
        errors += 1
        # Uncomment next line to see which dates fail and why:
        # print(f"  Skip {date.date()}: {e}")
        continue

# ── Build the zero curve dataframe ────────────────────────────
zero_curve_df = pd.DataFrame(zero_curve_records)

if zero_curve_df.empty:
    print("\n❌ PROBLEM: No zero curve dates were produced.")
    print("   This means ALL rows had issues. Check the next cell for diagnosis.")
else:
    zero_curve_df = zero_curve_df.set_index('date')
    zero_curve_df.index = pd.to_datetime(zero_curve_df.index)
    zero_curve_df = zero_curve_df.sort_index()

    # Save to disk
    zero_curve_df.to_csv(f'{BASE_DIR}/data/processed/zero_curve.csv')

    print(f"\n✅ BOOTSTRAPPING COMPLETE")
    print(f"   Dates successfully bootstrapped: {len(zero_curve_df)}")
    print(f"   Dates skipped (too few rates):   {skipped}")
    print(f"   Dates errored:                   {errors}")
    print(f"\n📊 Zero Rates (%) — Last 5 dates:")

    zero_cols = [c for c in zero_curve_df.columns if 'zero' in c]
    print(zero_curve_df[zero_cols].dropna(how='all').tail(5).to_string())

📊 Total dates in dataset:    3251
📊 Dates with ≥2 swap rates:  3251
📊 Dates skipped (too many NaN): 0

🔄 Bootstrapping zero curve... (this takes ~30-60 seconds)



Bootstrapping: 100%|██████████| 3251/3251 [00:03<00:00, 877.12dates/s]



✅ BOOTSTRAPPING COMPLETE
   Dates successfully bootstrapped: 3251
   Dates skipped (too few rates):   0
   Dates errored:                   0

📊 Zero Rates (%) — Last 5 dates:
            zero_2y  zero_5y  zero_10y  zero_30y
date                                            
2026-06-12   4.0523   4.3183    4.4593    5.5065
2026-06-15   4.0321   4.2861    4.4550    5.5013
2026-06-16   4.0119   4.2652    4.4111    5.4708
2026-06-17   4.1635   4.3776    4.4572    5.4241
2026-06-18   4.1635   4.3776    4.4572    5.4241


In [22]:
print(f"Length of zero_curve_records: {len(zero_curve_records)}")
display(zero_curve_records[:5]) # Display first 5 records if any

Length of zero_curve_records: 0


[]

In [28]:
# ============================================================
# CELL 6: CARRY AND ROLL-DOWN ANALYTICS
#
# CARRY: P&L from holding a swap for 1 month, assuming rates
#        are unchanged. For a receiver swap (long fixed), carry ≈
#        (fixed rate − overnight SOFR) × DV01 × (1/12)
#
# ROLL-DOWN: P&L from the passage of time on an upward-sloping
#        curve. A 5y swap "rolls" to become a 4y11m swap in 1 month.
#        If the curve is upward-sloping, it rolls to a lower yield,
#        generating P&L. Roll ≈ -slope × (1/12) × DV01
# ============================================================

def compute_dv01(tenor_y: float, par_rate: float,
                 payment_freq: int = 2) -> float:
    """
    DV01 (dollar value of 1bp) for a par swap per $1 notional.

    For a par swap, DV01 approximates the sum of discounted payment
    intervals weighted by maturity. Simplified closed-form:

        DV01 ≈ (1/payment_freq) × Σ D(t_i) × 0.0001

    where D(t_i) is the discount factor at each coupon date.
    The 0.0001 converts 1bp move to dollar terms.

    In practice this is the PV01 of the annuity. We use the
    approximation: DV01 ≈ tenor × 0.0001 / (1 + par_rate)^tenor
    (This is the Macaulay duration approach for a par bond.)
    """
    r = par_rate / 100
    # Modified duration for a par bond ≈ (1 - (1+r/f)^(-f*T)) / r
    f = payment_freq
    T = tenor_y
    if r == 0 or r != r:  # handle NaN/zero
        return tenor_y * 0.0001
    annuity = (1 - (1 + r/f)**(-f*T)) / (r/f)
    dv01 = annuity * 0.0001 / f   # per $1 notional, per bp
    return dv01


def compute_carry_rolldown(df: pd.DataFrame,
                            zero_curve_df: pd.DataFrame) -> pd.DataFrame:
    """
    For each tenor, compute:
    - carry_Xy: (par_swap_rate − SOFR) × (1/12), annualized in bps
    - rolldown_Xy: yield change from rolling 1M down the zero curve × DV01

    These represent the two main P&L components of a fixed-income
    position in the absence of yield changes (the 'carry' scenario).
    """
    merged = df.copy()
    tenors = {'2y': 2.0, '5y': 5.0, '10y': 10.0}
    swap_map = {'2y': 'swap_2y', '5y': 'swap_5y', '10y': 'swap_10y'}
    dv01_map  = {}

    results = {}

    for label, T in tenors.items():
        swap_col = swap_map[label]
        zero_col = f'zero_{int(T)}y'
        zero_col_short = f'zero_{int(T-1) if T > 1 else 1}y'  # roll 1 year shorter approx

        # ── Carry (bps/month) ────────────────────────────────
        # Receiver swap: earn the fixed rate, pay SOFR each period
        carry = ((merged[swap_col] - merged['sofr']) / 12)
        results[f'carry_{label}'] = carry

        # ── DV01 ─────────────────────────────────────────────
        dv01 = merged[swap_col].apply(lambda r: compute_dv01(T, r))
        results[f'dv01_{label}'] = dv01

        # ── Roll-Down (bps) ──────────────────────────────────
        # Estimate: how much yield changes as the tenor shortens by 1M
        # Roll ≈ (zero_rate[T] − zero_rate[T − 1/12]) ≈ slope / T × (1/12)
        # We approximate using T and T−1 year zero rates (available)
        if zero_col in zero_curve_df.columns:
            z_T  = zero_curve_df[zero_col].reindex(merged.index, method='ffill')
            if T > 2:
                z_short_col = f'zero_{int(T-1)}y' if f'zero_{int(T-1)}y' in zero_curve_df.columns else zero_col
                z_Tm1 = zero_curve_df[z_short_col].reindex(merged.index, method='ffill')
                annual_roll = z_T - z_Tm1       # bps per year difference
                monthly_roll = annual_roll / 12   # roll-down in 1 month
            else:
                monthly_roll = pd.Series(0.0, index=merged.index)
            results[f'rolldown_{label}'] = monthly_roll
        else:
            results[f'rolldown_{label}'] = pd.Series(0.0, index=merged.index)

    # ── Total carry + rolldown per tenor ─────────────────────
    for label in tenors:
        results[f'total_cr_{label}'] = (
            results[f'carry_{label}'] + results.get(f'rolldown_{label}', 0)
        )

    carry_df = pd.DataFrame(results, index=merged.index)
    carry_df.to_csv(f'{BASE_DIR}/data/processed/carry_rolldown.csv')

    print("✅ Carry and roll-down computed")
    print("\n📊 Latest Carry + Roll-Down (bps/month):")
    cr_cols = [c for c in carry_df.columns if 'total_cr' in c]
    print(carry_df[cr_cols].dropna().tail(3))

    return carry_df

carry_df = compute_carry_rolldown(df, zero_curve_df)

✅ Carry and roll-down computed

📊 Latest Carry + Roll-Down (bps/month):
            total_cr_2y  total_cr_5y  total_cr_10y
2026-06-16       0.0352       0.0444        0.0668
2026-06-17       0.0477       0.0535        0.0718
2026-06-18       0.0477       0.0535        0.0718


In [29]:
# ============================================================
# CELL 7: BUTTERFLY TRADE CONSTRUCTION
#
# Trade structure (standard market convention):
#   SHORT the belly (5y) — sell 5y swap (pay fixed)
#   LONG the wings (2y + 10y) — receive fixed
#
# DV01-neutral condition:
#   N_5y × DV01_5y = N_2y × DV01_2y + N_10y × DV01_10y
#
# Wing allocation: split evenly so N_2y = N_10y = w
#   w × (DV01_2y + DV01_10y) = N_5y × DV01_5y
#   → w = N_5y × DV01_5y / (DV01_2y + DV01_10y)
#
# The butterfly spread is then:
#   Butterfly = 2 × swap_5y − swap_2y − swap_10y  [bps]
#
# Economic intuition: if the belly is rich (spread > 0), we SHORT
# the belly expecting it to mean-revert toward the wings.
# ============================================================

def build_butterfly_strategy(df: pd.DataFrame,
                              carry_df: pd.DataFrame,
                              zscore_window: int = 252,
                              entry_z: float = 1.5,
                              exit_z: float = 0.5) -> pd.DataFrame:
    """
    Constructs the 2s5s10s butterfly strategy with:
    - DV01-neutral position sizing
    - Z-score signal (rolling 252-day mean/std)
    - Entry when |z| > entry_z, exit when |z| < exit_z
    - P&L decomposition: carry + roll-down + mark-to-market
    """
    strat = pd.DataFrame(index=df.index)

    # ── Butterfly Spread ─────────────────────────────────────
    strat['butterfly'] = (
        2 * df['swap_5y'] - df['swap_2y'] - df['swap_10y']
    ) * 100   # in bps

    strat['butterfly_tsy'] = (
        2 * df['tsy_5y'] - df['tsy_2y'] - df['tsy_10y']
    ) * 100

    # ── Z-Score Signal ───────────────────────────────────────
    roll_mean = strat['butterfly'].rolling(zscore_window, min_periods=126).mean()
    roll_std  = strat['butterfly'].rolling(zscore_window, min_periods=126).std()
    strat['z_score'] = (strat['butterfly'] - roll_mean) / roll_std

    # ── DV01s ─────────────────────────────────────────────────
    strat['dv01_2y']  = carry_df['dv01_2y']
    strat['dv01_5y']  = carry_df['dv01_5y']
    strat['dv01_10y'] = carry_df['dv01_10y']

    # ── DV01-Neutral Notional Ratios ─────────────────────────
    # Normalise so belly DV01 = 1 (per unit notional)
    # Wing sizes: w = DV01_5y / (DV01_2y + DV01_10y)
    strat['w_wings'] = strat['dv01_5y'] / (strat['dv01_2y'] + strat['dv01_10y'])

    # ── Trading Signal ────────────────────────────────────────
    # +1 = short belly (butterfly expensive → sell 5y, buy 2y+10y)
    # -1 = long belly  (butterfly cheap    → buy 5y, sell 2y+10y)
    #  0 = flat
    signal = pd.Series(0.0, index=strat.index)
    position = 0

    for i in range(1, len(strat)):
        z = strat['z_score'].iloc[i]
        if np.isnan(z):
            signal.iloc[i] = 0
            position = 0
            continue

        if position == 0:
            if z > entry_z:
                position = -1   # short belly (sell the expensive)
            elif z < -entry_z:
                position = 1    # long belly (buy the cheap)
        elif position == -1:
            if z < exit_z:
                position = 0
        elif position == 1:
            if z > -exit_z:
                position = 0

        signal.iloc[i] = position

    strat['signal'] = signal
    strat['position'] = signal.shift(1).fillna(0)  # trade next day open

    # ── P&L Decomposition ─────────────────────────────────────
    # Daily butterfly spread change → price P&L
    strat['spread_chg'] = strat['butterfly'].diff()

    # Carry + roll-down for the butterfly (2×carry_5y − carry_2y − carry_10y)
    # Adjusted for daily frequency (÷21 trading days per month)
    strat['butterfly_cr'] = (
        2 * carry_df['total_cr_5y']
        - carry_df.get('total_cr_2y', 0)
        - carry_df.get('total_cr_10y', 0)
    ) / 21   # monthly to daily

    # Total daily P&L per $1 notional
    # Short belly position profits when spread falls (negative change)
    strat['pnl_price']  = strat['position'] * (-strat['spread_chg'])
    strat['pnl_carry']  = strat['position'] * strat['butterfly_cr']
    strat['pnl_total']  = strat['pnl_price'] + strat['pnl_carry']
    strat['cum_pnl']    = strat['pnl_total'].cumsum()

    strat.to_csv(f'{BASE_DIR}/data/processed/butterfly_strategy.csv')

    print("✅ Butterfly strategy constructed")
    return strat

strat = build_butterfly_strategy(df, carry_df, zscore_window=252,
                                  entry_z=1.5, exit_z=0.5)
print(f"\n📊 Signal distribution:\n{strat['position'].value_counts()}")

✅ Butterfly strategy constructed

📊 Signal distribution:
position
0.0000     1481
1.0000      992
-1.0000     778
Name: count, dtype: int64


In [30]:
# ============================================================
# CELL 8: PERFORMANCE METRICS
# Institutional-grade risk/return analytics
# ============================================================

def compute_performance_metrics(strat: pd.DataFrame,
                                 risk_free_daily: float = 0.0) -> dict:
    """
    Compute Sharpe, Sortino, Calmar, max drawdown, hit rate,
    and P&L decomposition statistics.

    risk_free_daily: daily risk-free rate (default 0 for excess returns)
    """
    pnl = strat['pnl_total'].dropna()
    cum  = pnl.cumsum()

    # ── Annualised Return ─────────────────────────────────────
    n_years = len(pnl) / 252
    total_return = cum.iloc[-1]
    ann_return   = total_return / n_years

    # ── Sharpe Ratio ─────────────────────────────────────────
    excess = pnl - risk_free_daily
    sharpe = np.sqrt(252) * excess.mean() / excess.std()

    # ── Sortino Ratio ─────────────────────────────────────────
    downside = pnl[pnl < 0].std()
    sortino  = np.sqrt(252) * pnl.mean() / downside if downside > 0 else np.nan

    # ── Maximum Drawdown ──────────────────────────────────────
    roll_max  = cum.cummax()
    drawdown  = cum - roll_max
    max_dd    = drawdown.min()

    # Drawdown duration
    dd_duration = (drawdown < 0).sum()

    # ── Calmar Ratio ──────────────────────────────────────────
    calmar = ann_return / abs(max_dd) if max_dd != 0 else np.nan

    # ── Hit Rate ──────────────────────────────────────────────
    active_pnl = pnl[strat['position'].reindex(pnl.index) != 0]
    hit_rate   = (active_pnl > 0).mean()

    # ── P&L Decomposition ─────────────────────────────────────
    total_carry_pnl = strat['pnl_carry'].sum()
    total_price_pnl = strat['pnl_price'].sum()
    carry_pct = total_carry_pnl / total_return * 100 if total_return != 0 else 0

    metrics = {
        'Total Return (bps)':      round(total_return, 2),
        'Annualised Return (bps)': round(ann_return, 2),
        'Sharpe Ratio':            round(sharpe, 3),
        'Sortino Ratio':           round(sortino, 3),
        'Max Drawdown (bps)':      round(max_dd, 2),
        'Calmar Ratio':            round(calmar, 3),
        'Hit Rate':                f"{hit_rate:.1%}",
        'Carry P&L (bps)':        round(total_carry_pnl, 2),
        'Price P&L (bps)':        round(total_price_pnl, 2),
        'Carry % of Total':        f"{carry_pct:.1f}%",
        'Total Trades':            int((strat['position'].diff() != 0).sum()),
        'In-Market Days':          int((strat['position'] != 0).sum()),
    }

    print("=" * 50)
    print("  STRATEGY PERFORMANCE SUMMARY")
    print("=" * 50)
    for k, v in metrics.items():
        print(f"  {k:<30} {v}")
    print("=" * 50)

    return metrics

metrics = compute_performance_metrics(strat)

  STRATEGY PERFORMANCE SUMMARY
  Total Return (bps)             -47.62
  Annualised Return (bps)        -5.6
  Sharpe Ratio                   -0.174
  Sortino Ratio                  -0.198
  Max Drawdown (bps)             -87.2
  Calmar Ratio                   -0.064
  Hit Rate                       45.4%
  Carry P&L (bps)                -0.62
  Price P&L (bps)                -84.0
  Carry % of Total               1.3%
  Total Trades                   58
  In-Market Days                 1770


In [31]:
# ============================================================
# CELL 9: VISUALIZATION 1 — SWAP SPREAD TIME SERIES
# Shows 2y, 5y, 10y swap spreads over time with crisis shading
# ============================================================

def plot_swap_spreads(df: pd.DataFrame) -> go.Figure:
    """
    Interactive Plotly chart of swap spreads (bps).
    Zero-line reference: when spread < 0, swaps trade inside Treasuries
    (a structural anomaly that has occurred post-GFC).
    """
    fig = make_subplots(
        rows=2, cols=1,
        subplot_titles=('USD Swap Spreads (bps)', 'Treasury Yield Curve Slope (10s2s, bps)'),
        vertical_spacing=0.12,
        row_heights=[0.65, 0.35]
    )

    colors = {'ss_2y': '#1f77b4', 'ss_5y': '#ff7f0e', 'ss_10y': '#2ca02c'}
    labels = {'ss_2y': '2Y Swap Spread', 'ss_5y': '5Y Swap Spread', 'ss_10y': '10Y Swap Spread'}

    for col, color in colors.items():
        s = df[col].dropna()
        fig.add_trace(go.Scatter(
            x=s.index, y=s.values,
            name=labels[col],
            line=dict(color=color, width=1.5),
            hovertemplate='%{x|%Y-%m-%d}<br>%{y:.1f} bps<extra></extra>'
        ), row=1, col=1)

    # Zero reference line
    fig.add_hline(y=0, line_dash='dash', line_color='red',
                  annotation_text='Zero (swaps = treasuries)',
                  row=1, col=1)

    # Slope panel
    slope = df['slope_bps'].dropna()
    fig.add_trace(go.Scatter(
        x=slope.index, y=slope.values,
        name='10s2s Slope',
        line=dict(color='#9467bd', width=1.5),
        fill='tozeroy', fillcolor='rgba(148, 103, 189, 0.15)'
    ), row=2, col=1)

    fig.update_layout(
        title=dict(
            text='<b>USD Interest Rate Swap Spreads</b><br>'
                 '<sup>Swap Rate − Treasury Yield | Proxy for Bank Credit/Funding Risk Premium</sup>',
            font=dict(size=16)
        ),
        height=700,
        template='plotly_white',
        legend=dict(orientation='h', y=1.02, x=0),
        hovermode='x unified'
    )
    fig.update_yaxes(title_text='Basis Points (bps)', row=1, col=1)
    fig.update_yaxes(title_text='bps', row=2, col=1)

    fig.write_html(f'{BASE_DIR}/outputs/charts/swap_spreads.html')
    fig.show()
    return fig

fig1 = plot_swap_spreads(df)
print("✅ Swap spreads chart saved")

✅ Swap spreads chart saved


In [32]:
# ============================================================
# CELL 10: VISUALIZATION 2 — 2s5s10s BUTTERFLY WITH SIGNALS
# ============================================================

def plot_butterfly_signals(strat: pd.DataFrame) -> go.Figure:
    """
    Three-panel chart:
    1. Butterfly spread with entry/exit signal bands
    2. Z-score with threshold lines
    3. Position (long belly / short belly / flat)
    """
    fig = make_subplots(
        rows=3, cols=1,
        subplot_titles=(
            '2s5s10s Swap Butterfly Spread (bps)',
            'Z-Score (252-day rolling)',
            'Strategy Position'
        ),
        vertical_spacing=0.08,
        row_heights=[0.45, 0.30, 0.25]
    )

    # ── Panel 1: Butterfly Spread ─────────────────────────────
    fly = strat['butterfly'].dropna()
    fig.add_trace(go.Scatter(
        x=fly.index, y=fly.values,
        name='2s5s10s Butterfly',
        line=dict(color='#1f77b4', width=1.5),
        hovertemplate='%{x|%Y-%m-%d}<br>%{y:.2f} bps<extra></extra>'
    ), row=1, col=1)

    # Rolling mean band
    roll_m = strat['butterfly'].rolling(252, min_periods=126).mean().dropna()
    roll_s = strat['butterfly'].rolling(252, min_periods=126).std().dropna()
    fig.add_trace(go.Scatter(
        x=roll_m.index, y=roll_m.values,
        name='Rolling Mean', line=dict(color='gray', dash='dash', width=1)
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=roll_m.index, y=(roll_m + 1.5*roll_s).values,
        name='+1.5σ', line=dict(color='red', dash='dot', width=0.8)
    ), row=1, col=1)
    fig.add_trace(go.Scatter(
        x=roll_m.index, y=(roll_m - 1.5*roll_s).values,
        name='−1.5σ', line=dict(color='green', dash='dot', width=0.8),
        fill='tonexty', fillcolor='rgba(200, 200, 200, 0.15)'
    ), row=1, col=1)

    # ── Panel 2: Z-Score ──────────────────────────────────────
    z = strat['z_score'].dropna()
    fig.add_trace(go.Scatter(
        x=z.index, y=z.values,
        name='Z-Score',
        line=dict(color='#ff7f0e', width=1.5)
    ), row=2, col=1)
    for level, color in [(1.5, 'red'), (-1.5, 'green'), (0, 'gray')]:
        fig.add_hline(y=level, line_dash='dash', line_color=color,
                      line_width=0.8, row=2, col=1)

    # ── Panel 3: Position ─────────────────────────────────────
    pos = strat['position'].dropna()
    colors_pos = pos.map({1: 'green', -1: 'red', 0: 'gray'})
    fig.add_trace(go.Bar(
        x=pos.index, y=pos.values,
        name='Position',
        marker_color=colors_pos,
        hovertemplate='%{x|%Y-%m-%d}<br>Position: %{y}<extra></extra>'
    ), row=3, col=1)

    fig.update_layout(
        title=dict(
            text='<b>2s5s10s Swap Butterfly Strategy</b><br>'
                 '<sup>Short Belly when Z > +1.5σ | Long Belly when Z < −1.5σ | Exit at |Z| < 0.5σ</sup>',
            font=dict(size=16)
        ),
        height=850,
        template='plotly_white',
        showlegend=True,
        legend=dict(orientation='h', y=1.02, x=0),
        hovermode='x unified'
    )

    fig.write_html(f'{BASE_DIR}/outputs/charts/butterfly_signals.html')
    fig.show()
    return fig

fig2 = plot_butterfly_signals(strat)
print("✅ Butterfly signals chart saved")

✅ Butterfly signals chart saved


In [33]:
# ============================================================
# CELL 11: VISUALIZATION 3 — CARRY + ROLL-DOWN DECOMPOSITION
# ============================================================

def plot_carry_rolldown(carry_df: pd.DataFrame) -> go.Figure:
    """
    Stacked bar chart showing carry and roll-down contributions
    for each tenor. Provides intuition for which part of the curve
    offers the best risk-adjusted carry profile.
    """
    # Resample to monthly for readability
    monthly = carry_df[['carry_2y', 'carry_5y', 'carry_10y',
                          'rolldown_2y', 'rolldown_5y', 'rolldown_10y']].resample('ME').last()
    monthly = monthly.dropna()

    fig = make_subplots(
        rows=1, cols=3,
        subplot_titles=['2Y Swap', '5Y Swap', '10Y Swap'],
        shared_yaxes=True
    )

    configs = [
        ('2y', 1), ('5y', 2), ('10y', 3)
    ]
    colors_carry   = ['#1f77b4', '#ff7f0e', '#2ca02c']
    colors_rolldown = ['#aec7e8', '#ffbb78', '#98df8a']

    for i, (label, col) in enumerate(configs):
        fig.add_trace(go.Bar(
            x=monthly.index, y=monthly[f'carry_{label}'],
            name=f'Carry {label.upper()}',
            marker_color=colors_carry[i],
            legendgroup=label,
            showlegend=True
        ), row=1, col=col)

        fig.add_trace(go.Bar(
            x=monthly.index, y=monthly.get(f'rolldown_{label}',
                                             pd.Series(0, index=monthly.index)),
            name=f'Roll-Down {label.upper()}',
            marker_color=colors_rolldown[i],
            legendgroup=label
        ), row=1, col=col)

    fig.update_layout(
        barmode='stack',
        title=dict(
            text='<b>Carry + Roll-Down Decomposition by Tenor</b><br>'
                 '<sup>Monthly bps | Carry = (Swap Rate − SOFR)/12 | Roll = Yield Change as Tenor Shortens</sup>',
            font=dict(size=16)
        ),
        height=500,
        template='plotly_white',
        hovermode='x unified'
    )
    fig.update_yaxes(title_text='bps/month', row=1, col=1)

    fig.write_html(f'{BASE_DIR}/outputs/charts/carry_rolldown.html')
    fig.show()
    return fig

fig3 = plot_carry_rolldown(carry_df)
print("✅ Carry/roll-down chart saved")

✅ Carry/roll-down chart saved


In [34]:
# ============================================================
# CELL 12: VISUALIZATION 4 — CUMULATIVE P&L BACKTEST
# ============================================================

def plot_backtest_pnl(strat: pd.DataFrame, metrics: dict) -> go.Figure:
    """
    Cumulative P&L with carry/price decomposition and drawdown.
    This is the primary performance attribution chart.
    """
    fig = make_subplots(
        rows=2, cols=1,
        subplot_titles=(
            'Cumulative P&L: Butterfly Strategy (bps)',
            'Drawdown (bps)'
        ),
        vertical_spacing=0.12,
        row_heights=[0.65, 0.35]
    )

    cum_total = strat['pnl_total'].cumsum().dropna()
    cum_carry = strat['pnl_carry'].cumsum().dropna()
    cum_price = strat['pnl_price'].cumsum().dropna()

    # ── Cumulative P&L components ────────────────────────────
    fig.add_trace(go.Scatter(
        x=cum_total.index, y=cum_total.values,
        name='Total P&L',
        line=dict(color='#1f77b4', width=2.5),
        fill='tozeroy', fillcolor='rgba(31, 119, 180, 0.1)'
    ), row=1, col=1)

    fig.add_trace(go.Scatter(
        x=cum_carry.index, y=cum_carry.values,
        name='Carry P&L',
        line=dict(color='#2ca02c', width=1.5, dash='dash')
    ), row=1, col=1)

    fig.add_trace(go.Scatter(
        x=cum_price.index, y=cum_price.values,
        name='Price P&L',
        line=dict(color='#ff7f0e', width=1.5, dash='dot')
    ), row=1, col=1)

    # Annotate key metrics on chart
    last_x = cum_total.index[-1]
    fig.add_annotation(
        x=last_x, y=cum_total.iloc[-1],
        text=f"Sharpe: {metrics['Sharpe Ratio']:.2f}<br>"
             f"MaxDD: {metrics['Max Drawdown (bps)']:.1f} bps<br>"
             f"Hit Rate: {metrics['Hit Rate']}",
        showarrow=True, arrowhead=1,
        bgcolor='white', bordercolor='gray',
        row=1, col=1
    )

    # ── Drawdown ──────────────────────────────────────────────
    cum_max = cum_total.cummax()
    drawdown = cum_total - cum_max

    fig.add_trace(go.Scatter(
        x=drawdown.index, y=drawdown.values,
        name='Drawdown',
        line=dict(color='red', width=1),
        fill='tozeroy', fillcolor='rgba(255, 0, 0, 0.2)'
    ), row=2, col=1)

    fig.update_layout(
        title=dict(
            text='<b>2s5s10s Butterfly Strategy Backtest</b><br>'
                 '<sup>Z-Score Mean Reversion | Entry ±1.5σ | Exit ±0.5σ</sup>',
            font=dict(size=16)
        ),
        height=700,
        template='plotly_white',
        legend=dict(orientation='h', y=1.02, x=0),
        hovermode='x unified'
    )
    fig.update_yaxes(title_text='Cumulative bps', row=1, col=1)
    fig.update_yaxes(title_text='Drawdown (bps)', row=2, col=1)

    fig.write_html(f'{BASE_DIR}/outputs/charts/backtest_pnl.html')
    fig.show()
    return fig

fig4 = plot_backtest_pnl(strat, metrics)
print("✅ Backtest P&L chart saved")

✅ Backtest P&L chart saved


In [35]:
# ============================================================
# CELL 13: EXCEL EXPORT — PROFESSIONAL DELIVERABLE
# Produces a multi-tab Excel workbook for portfolio presentation
# ============================================================

from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.chart import LineChart, Reference

def export_to_excel(df, zero_curve_df, carry_df, strat, metrics):
    """
    Creates a professional Excel workbook with:
    1. Summary dashboard
    2. Swap spreads data
    3. Zero curve data
    4. Carry/roll-down data
    5. Strategy signals & P&L
    6. Performance metrics
    """
    import openpyxl
    from openpyxl.styles import Font, PatternFill, Alignment

    wb = openpyxl.Workbook()

    # ── Color scheme ─────────────────────────────────────────
    HEADER_FILL = PatternFill("solid", fgColor="003366")
    HEADER_FONT = Font(bold=True, color="FFFFFF", size=11)
    ALT_FILL    = PatternFill("solid", fgColor="EBF1F8")

    def style_header_row(ws, row_num, num_cols):
        for col in range(1, num_cols + 1):
            cell = ws.cell(row=row_num, column=col)
            cell.fill  = HEADER_FILL
            cell.font  = HEADER_FONT
            cell.alignment = Alignment(horizontal='center')

    # ── Sheet 1: Performance Summary ─────────────────────────
    ws1 = wb.active
    ws1.title = "Summary"
    ws1['A1'] = "Fixed Income RV: Swap Spread & Butterfly Analyzer"
    ws1['A1'].font = Font(bold=True, size=14, color="003366")
    ws1['A2'] = f"Generated: {datetime.today().strftime('%Y-%m-%d')}"
    ws1['A3'] = "Author: Shashank Ravi | MSc Economics & Finance, KCL"

    ws1['A5'] = "PERFORMANCE METRICS"
    ws1['A5'].font = Font(bold=True, size=12)
    style_header_row(ws1, 6, 2)
    ws1['A6'], ws1['B6'] = "Metric", "Value"

    for i, (k, v) in enumerate(metrics.items(), start=7):
        ws1[f'A{i}'] = k
        ws1[f'B{i}'] = str(v)
        if i % 2 == 0:
            ws1[f'A{i}'].fill = ALT_FILL
            ws1[f'B{i}'].fill = ALT_FILL

    ws1.column_dimensions['A'].width = 30
    ws1.column_dimensions['B'].width = 20

    # ── Sheet 2: Swap Spreads ─────────────────────────────────
    ws2 = wb.create_sheet("Swap Spreads")
    spread_cols = ['swap_2y', 'swap_5y', 'swap_10y',
                   'tsy_2y',  'tsy_5y',  'tsy_10y',
                   'ss_2y',   'ss_5y',   'ss_10y', 'butterfly_swap']
    ss_df = df[spread_cols].dropna().reset_index()
    ss_df.columns = ['Date', 'Swap 2Y (%)', 'Swap 5Y (%)', 'Swap 10Y (%)',
                     'TSY 2Y (%)', 'TSY 5Y (%)', 'TSY 10Y (%)',
                     'SS 2Y (bps)', 'SS 5Y (bps)', 'SS 10Y (bps)', 'Butterfly (bps)']
    for r_idx, row in enumerate(dataframe_to_rows(ss_df, index=False, header=True), 1):
        ws2.append(row)
    style_header_row(ws2, 1, len(ss_df.columns))

    # ── Sheet 3: Zero Curve ───────────────────────────────────
    ws3 = wb.create_sheet("Zero Curve")
    zc_df = zero_curve_df.reset_index()
    for r_idx, row in enumerate(dataframe_to_rows(zc_df, index=False, header=True), 1):
        ws3.append(row)
    style_header_row(ws3, 1, len(zc_df.columns))

    # ── Sheet 4: Strategy Signals ─────────────────────────────
    ws4 = wb.create_sheet("Strategy")
    sig_cols = ['butterfly', 'z_score', 'signal', 'position',
                'pnl_carry', 'pnl_price', 'pnl_total', 'cum_pnl']
    sig_df = strat[sig_cols].dropna().reset_index()
    for r_idx, row in enumerate(dataframe_to_rows(sig_df, index=False, header=True), 1):
        ws4.append(row)
    style_header_row(ws4, 1, len(sig_df.columns))

    # ── Sheet 5: Carry/Roll-Down ──────────────────────────────
    ws5 = wb.create_sheet("Carry Roll-Down")
    cr_df = carry_df.reset_index()
    for r_idx, row in enumerate(dataframe_to_rows(cr_df, index=False, header=True), 1):
        ws5.append(row)
    style_header_row(ws5, 1, len(cr_df.columns))

    # Auto-fit columns for all sheets
    for ws in [ws2, ws3, ws4, ws5]:
        for col in ws.columns:
            max_len = max(len(str(cell.value)) if cell.value else 0 for cell in col)
            ws.column_dimensions[col[0].column_letter].width = min(max_len + 2, 25)

    path = f'{BASE_DIR}/outputs/reports/fixed_income_rv_report.xlsx'
    wb.save(path)
    print(f"✅ Excel report saved → {path}")
    return path

export_to_excel(df, zero_curve_df, carry_df, strat, metrics)

✅ Excel report saved → /content/fixed_income_rv/outputs/reports/fixed_income_rv_report.xlsx


'/content/fixed_income_rv/outputs/reports/fixed_income_rv_report.xlsx'

In [36]:
# ============================================================
# CELL 14: FINAL SUMMARY & FILE DOWNLOAD
# ============================================================

from google.colab import files
import zipfile

print("=" * 60)
print("  FIXED INCOME RV: PROJECT COMPLETE")
print("=" * 60)
print(f"\n📊 Data period:        {df.index[0].date()} → {df.index[-1].date()}")
print(f"📈 Zero curve dates:   {len(zero_curve_df)}")
print(f"📉 Strategy signals:   {int((strat['position'] != 0).sum())} active days")
print(f"\n📁 Output files generated:")

output_files = []
for root, dirs, files_list in os.walk(f'{BASE_DIR}/outputs'):
    for f_name in files_list:
        fpath = os.path.join(root, f_name)
        fsize = os.path.getsize(fpath) / 1024
        print(f"   {fpath.replace(BASE_DIR+'/', '')} ({fsize:.0f} KB)")
        output_files.append(fpath)

# Zip everything for download
zip_path = '/content/fixed_income_rv_outputs.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fpath in output_files:
        zf.write(fpath, fpath.replace('/content/', ''))

print(f"\n📦 All outputs zipped → {zip_path}")

# Download
files.download(zip_path)

  FIXED INCOME RV: PROJECT COMPLETE

📊 Data period:        2014-01-02 → 2026-06-18
📈 Zero curve dates:   3251
📉 Strategy signals:   1770 active days

📁 Output files generated:
   outputs/reports/fixed_income_rv_report.xlsx (692 KB)
   outputs/charts/carry_rolldown.html (4481 KB)
   outputs/charts/butterfly_signals.html (5199 KB)
   outputs/charts/backtest.html (4719 KB)
   outputs/charts/dashboard.html (4940 KB)
   outputs/charts/swap_spreads.html (4945 KB)
   outputs/charts/backtest_pnl.html (4850 KB)

📦 All outputs zipped → /content/fixed_income_rv_outputs.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [37]:
# ============================================================
# CELL A: INSTALL XCCY EXTENSION LIBRARIES
# No new accounts needed — ECB and BoE APIs are fully public
# ============================================================

!pip install requests lxml xmltodict -q
print("✅ XCCY extension libraries installed!")

✅ XCCY extension libraries installed!


In [38]:
# ============================================================
# CELL B: ECB DATA PORTAL — EUR SWAP RATES
#
# The ECB provides free SDMX REST API access.
# No API key required. We hit a URL and get back data.
#
# ECB SDMX Series Key format:
# YC.B.U2.EUR.4F.G_N_A.SV_C_YM.SR_10Y
#   YC       = Yield Curve
#   B        = Business
#   U2       = Euro area
#   EUR      = Euro currency
#   4F       = ECB methodology
#   G_N_A    = Government bonds
#   SV_C_YM  = Svensson model, continuous, par yield
#   SR_10Y   = Spot rate, 10 year
#
# For SWAP (IRS) rates we use the FM (Financial Markets) dataflow
# Series: FM.B.U2.EUR.FR.BB.EURIBOR3MD_.  ← EURIBOR-based
# We use EUR OIS (ESTR) swap rates which are the modern standard
# ============================================================

import requests
import pandas as pd
import numpy as np
from datetime import datetime
import io

BASE_DIR = '/content/fixed_income_rv'   # same as before

def fetch_ecb_series(series_key: str, start: str, end: str,
                      label: str) -> pd.Series:
    """
    Fetch a single time series from the ECB SDMX REST API.

    URL format:
    https://data-api.ecb.europa.eu/service/data/{DATAFLOW}/{SERIES_KEY}
    ?startPeriod=YYYY-MM-DD&endPeriod=YYYY-MM-DD&format=csvdata

    No authentication required. Returns a pandas Series.
    """
    # ECB uses different dataflows for different data types
    # For yield curve / swap rates, we use the YC dataflow
    url = (
        f"https://data-api.ecb.europa.eu/service/data/{series_key}"
        f"?startPeriod={start}&endPeriod={end}&format=csvdata&detail=dataonly"
    )

    try:
        response = requests.get(url, timeout=30,
                                 headers={'Accept': 'text/csv'})
        response.raise_for_status()
        df = pd.read_csv(io.StringIO(response.text))

        # ECB CSV has columns: KEY, FREQ, etc., plus TIME_PERIOD and OBS_VALUE
        if 'TIME_PERIOD' in df.columns and 'OBS_VALUE' in df.columns:
            s = df.set_index('TIME_PERIOD')['OBS_VALUE']
            s.index = pd.to_datetime(s.index)
            s = pd.to_numeric(s, errors='coerce')
            s.name = label
            print(f"  ✅ ECB {label}: {len(s)} obs | {s.index[0].date()} → {s.index[-1].date()}")
            return s
        else:
            print(f"  ❌ ECB {label}: Unexpected column structure")
            print(f"     Columns found: {list(df.columns[:5])}")
            return pd.Series(dtype=float, name=label)

    except Exception as e:
        print(f"  ❌ ECB {label}: {e}")
        return pd.Series(dtype=float, name=label)


# ── ECB Series Keys ──────────────────────────────────────────
# EUR AAA government bond par yields (best proxy for EUR risk-free)
# ECB publishes these daily via the YC (Yield Curve) dataflow
ECB_SERIES = {
    'eur_2y':  'YC/B.U2.EUR.4F.G_N_A.SV_C_YM.SR_2Y',
    'eur_5y':  'YC/B.U2.EUR.4F.G_N_A.SV_C_YM.SR_5Y',
    'eur_10y': 'YC/B.U2.EUR.4F.G_N_A.SV_C_YM.SR_10Y',
    'eur_30y': 'YC/B.U2.EUR.4F.G_N_A.SV_C_YM.SR_30Y',
}

START = '2014-01-01'
END   = datetime.today().strftime('%Y-%m-%d')

print("🔄 Fetching EUR yield data from ECB Data Portal (no key needed)...")
ecb_frames = {}
for name, key in ECB_SERIES.items():
    ecb_frames[name] = fetch_ecb_series(key, START, END, name)

eur_df = pd.DataFrame(ecb_frames)
eur_df.index = pd.to_datetime(eur_df.index)
eur_df = eur_df.sort_index()

# ── Fallback: If ECB API returns empty, use FRED EUR series ──
# FRED has some EUR government bond yields as backup
# (less comprehensive but robust)
from fredapi import Fred
fred = Fred(api_key=FRED_API_KEY)   # reuse key from earlier cells

if eur_df.isnull().all().all():
    print("\n⚠️  ECB API returned no data. Using FRED EUR backup...")
    # FRED EUR 10Y government benchmark
    try:
        eur10_backup = fred.get_series('IRLTLT01EZM156N',
                                        observation_start=START,
                                        observation_end=END)
        eur_df['eur_10y_backup'] = eur10_backup
        print("  ✅ FRED EUR 10Y backup loaded")
    except Exception as e:
        print(f"  ❌ FRED EUR backup also failed: {e}")

eur_df.to_csv(f'{BASE_DIR}/data/raw/eur_rates.csv')
print(f"\n📦 EUR data shape: {eur_df.shape}")
eur_df.tail(3)

🔄 Fetching EUR yield data from ECB Data Portal (no key needed)...
  ✅ ECB eur_2y: 3178 obs | 2014-01-02 → 2026-06-18
  ✅ ECB eur_5y: 3178 obs | 2014-01-02 → 2026-06-18
  ✅ ECB eur_10y: 3178 obs | 2014-01-02 → 2026-06-18
  ✅ ECB eur_30y: 3178 obs | 2014-01-02 → 2026-06-18

📦 EUR data shape: (3178, 4)


,eur_2y,eur_5y,eur_10y,eur_30y
TIME_PERIOD,,,,
2026-06-16,2.5154,2.6414,2.9924,3.4979
2026-06-17,2.5267,2.6383,2.9855,3.4783
2026-06-18,2.5544,2.6554,2.9853,3.4663


In [41]:
# ============================================================
# CELL C (FIXED): GBP YIELD DATA — THREE METHODS, AUTO-FALLBACK
#
# WHY IT FAILED:
# The BoE website blocks Python's default requests user-agent.
# It sees "python-requests/2.x" and returns 403 Forbidden.
# Fix: Spoof a real browser User-Agent header in the request.
#
# STRUCTURE:
# Method 1 → BoE API with browser headers (primary)
# Method 2 → BoE direct CSV download (backup)
# Method 3 → FRED UK gilt series (bulletproof fallback)
# ============================================================

import requests
import pandas as pd
import numpy as np
import io
from datetime import datetime

BASE_DIR = '/content/fixed_income_rv'
START = '2014-01-01'
END   = datetime.today().strftime('%Y-%m-%d')


# ─────────────────────────────────────────────────────────────
# METHOD 1: BoE API WITH BROWSER HEADERS
# The 403 error happens because BoE checks the User-Agent.
# We spoof a real Chrome browser header to get through.
# ─────────────────────────────────────────────────────────────

def fetch_boe_with_headers(series_codes: list,
                            start_date: str,
                            end_date: str) -> pd.DataFrame:
    """
    Fetch BoE data using a browser-like User-Agent to bypass the 403.
    The BoE site checks whether the requester looks like a real browser.
    """
    from datetime import datetime as dt

    # Convert dates to BoE format: DD/Mon/YYYY
    start_boe = dt.strptime(start_date, '%Y-%m-%d').strftime('%d/%b/%Y')
    end_boe   = dt.strptime(end_date,   '%Y-%m-%d').strftime('%d/%b/%Y')

    codes_str = ','.join(series_codes)

    url = (
        "https://www.bankofengland.co.uk/boeapps/iadb/fromshowcolumns.asp"
        f"?csv.x=yes"
        f"&Datefrom={start_boe}"
        f"&Dateto={end_boe}"
        f"&SeriesCodes={codes_str}"
        f"&CSVF=TN&UsingCodes=Y&VPD=Y&VFD=N"
    )

    # ── THE FIX: Full browser headers ────────────────────────
    # This is exactly what Chrome sends when you visit a website.
    # Without this, BoE returns 403. With it, it thinks you're a browser.
    headers = {
        'User-Agent': (
            'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
            'AppleWebKit/537.36 (KHTML, like Gecko) '
            'Chrome/124.0.0.0 Safari/537.36'
        ),
        'Accept': 'text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8',
        'Accept-Language': 'en-GB,en;q=0.9',
        'Referer': 'https://www.bankofengland.co.uk/statistics',
        'Connection': 'keep-alive',
    }

    # Use a Session object — more reliable than plain requests.get()
    session = requests.Session()
    session.headers.update(headers)

    response = session.get(url, timeout=30)
    response.raise_for_status()   # raises an error if still 403/404

    # BoE returns tab-separated data
    # First row is headers, DATE column is in DD Mon YYYY format
    df = pd.read_csv(
        io.StringIO(response.text),
        sep='\t',
        parse_dates=['DATE'],
        dayfirst=True
    )

    df = df.set_index('DATE')
    df.index = pd.to_datetime(df.index, dayfirst=True, errors='coerce')
    df = df[df.index.notna()]   # drop rows where date parsing failed

    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    df = df.sort_index()
    return df


# ─────────────────────────────────────────────────────────────
# METHOD 2: BoE YIELD CURVE CSV DIRECT DOWNLOAD
# The BoE also publishes yield curve data as a direct CSV file.
# Different URL format — sometimes works when the API doesn't.
# Series: instantaneous nominal forward curve (fitted, daily)
# ─────────────────────────────────────────────────────────────

def fetch_boe_yield_curve_csv() -> pd.DataFrame:
    """
    Download the BoE's published fitted gilt yield curve directly.
    This is the Svensson/VRP model curve published daily.
    URL format confirmed working as of 2025.
    """
    # BoE publishes their fitted yield curve as a downloadable CSV
    # This is more stable than the iadb API
    url = "https://www.bankofengland.co.uk/statistics/yield-curves"

    # They also publish a direct Excel/CSV via this endpoint
    # The spot curve data file:
    yield_curve_url = (
        "https://www.bankofengland.co.uk"
        "/boeapps/iadb/fromshowcolumns.asp"
        "?csv.x=yes&Datefrom=02/Jan/2014"
        f"&Dateto={datetime.today().strftime('%d/%b/%Y')}"
        "&SeriesCodes=IUDSNPY2,IUDSNPY5,IUDSNPY10,IUDSNPY20,IUDSOIA"
        "&CSVF=TN&UsingCodes=Y&VPD=Y&VFD=N"
    )

    headers = {
        'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) '
                      'AppleWebKit/537.36 (KHTML, like Gecko) '
                      'Chrome/124.0.0.0 Safari/537.36',
        'Referer': 'https://www.bankofengland.co.uk/',
        'Accept-Language': 'en-GB,en;q=0.9',
    }

    session = requests.Session()
    session.headers.update(headers)

    # First hit the stats page to get a session cookie (important!)
    session.get('https://www.bankofengland.co.uk/statistics',
                timeout=15)

    # Now request the actual data
    response = session.get(yield_curve_url, timeout=30)
    response.raise_for_status()

    df = pd.read_csv(
        io.StringIO(response.text),
        sep='\t', parse_dates=['DATE'], dayfirst=True
    )
    df = df.set_index('DATE')
    df.index = pd.to_datetime(df.index, dayfirst=True, errors='coerce')
    df = df[df.index.notna()]

    for col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

    return df.sort_index()


# ─────────────────────────────────────────────────────────────
# METHOD 3: FRED UK GILT YIELDS (BULLETPROOF FALLBACK)
# FRED hosts UK government bond yield data sourced from the
# OECD. Not as granular as BoE but completely reliable.
#
# FRED Series:
#   IRLTLT01GBM156N  = UK 10Y gilt yield (monthly, OECD)
#   For 2Y and 5Y, we use the BoE/OECD series available on FRED
# ─────────────────────────────────────────────────────────────

def fetch_gbp_from_fred(fred_client) -> pd.DataFrame:
    """
    Pull GBP gilt yields from FRED as a fallback.
    These are monthly series — we'll forward-fill to daily.
    """
    fred_gbp_series = {
        'gbp_10y': 'IRLTLT01GBM156N',   # UK 10Y long-term rate (monthly)
        'gbp_2y':  'IR3TIB01GBM156N',   # UK 3M interbank (best FRED proxy for short end)
    }

    frames = {}
    for name, ticker in fred_gbp_series.items():
        try:
            s = fred_client.get_series(
                ticker,
                observation_start=START,
                observation_end=END
            )
            frames[name] = s
            print(f"  ✅ FRED {ticker} → {name}: {len(s)} obs")
        except Exception as e:
            print(f"  ⚠️  FRED {ticker} failed: {e}")

    if not frames:
        return pd.DataFrame()

    df = pd.DataFrame(frames)
    df.index = pd.to_datetime(df.index)
    # Resample to daily and forward fill (monthly → daily)
    df = df.resample('B').interpolate(method='linear')
    df = df.ffill(limit=5)

    # Interpolate 5Y as midpoint between 2Y and 10Y
    if 'gbp_2y' in df.columns and 'gbp_10y' in df.columns:
        df['gbp_5y'] = df['gbp_2y'] * 0.4 + df['gbp_10y'] * 0.6
        df['gbp_30y'] = df['gbp_10y'] + 0.4   # rough 30Y proxy

    # SONIA fallback: approximate from FRED SONIA series or BoE rate
    try:
        sonia = fred_client.get_series('IUDSOIA',
                                        observation_start=START,
                                        observation_end=END)
        df['sonia'] = sonia.reindex(df.index, method='ffill')
    except:
        # Use BoE base rate as SONIA proxy
        try:
            boe_rate = fred_client.get_series('BOERUKM',
                                               observation_start=START,
                                               observation_end=END)
            df['sonia'] = boe_rate.reindex(df.index, method='ffill')
        except:
            df['sonia'] = np.nan

    return df


# ─────────────────────────────────────────────────────────────
# MAIN: TRY EACH METHOD IN ORDER, USE FIRST THAT WORKS
# ─────────────────────────────────────────────────────────────

BOE_SERIES = ['IUDSNPY2', 'IUDSNPY5', 'IUDSNPY10', 'IUDSNPY20', 'IUDSOIA']

RENAME_BOE = {
    'IUDSNPY2':  'gbp_2y',
    'IUDSNPY5':  'gbp_5y',
    'IUDSNPY10': 'gbp_10y',
    'IUDSNPY20': 'gbp_30y',
    'IUDSOIA':   'sonia',
}

gbp_df = pd.DataFrame()
method_used = None

# ── Try Method 1: BoE with browser headers ────────────────────
print("🔄 Method 1: BoE API with browser headers...")
try:
    raw = fetch_boe_with_headers(BOE_SERIES, START, END)
    if len(raw) > 100:
        gbp_df = raw.rename(columns=RENAME_BOE)
        method_used = 'BoE API (browser headers)'
        print(f"  ✅ Success! {len(gbp_df)} rows")
    else:
        print(f"  ⚠️  Returned only {len(raw)} rows — trying next method")
except Exception as e:
    print(f"  ❌ Failed: {e}")

# ── Try Method 2: BoE with session cookie ─────────────────────
if gbp_df.empty:
    print("\n🔄 Method 2: BoE with session cookie...")
    try:
        raw = fetch_boe_yield_curve_csv()
        if len(raw) > 100:
            gbp_df = raw.rename(columns=RENAME_BOE)
            method_used = 'BoE yield curve CSV (session)'
            print(f"  ✅ Success! {len(gbp_df)} rows")
        else:
            print(f"  ⚠️  Still only {len(raw)} rows — trying FRED fallback")
    except Exception as e:
        print(f"  ❌ Failed: {e}")

# ── Method 3: FRED (always works) ─────────────────────────────
if gbp_df.empty:
    print("\n🔄 Method 3: FRED UK gilt data (reliable fallback)...")
    try:
        gbp_df = fetch_gbp_from_fred(fred)   # 'fred' from your earlier cells
        if not gbp_df.empty:
            method_used = 'FRED UK gilt series'
            print(f"  ✅ Success via FRED! {len(gbp_df)} rows")
        else:
            print("  ❌ FRED also returned empty — check your API key")
    except Exception as e:
        print(f"  ❌ FRED failed: {e}")

# ── Final status ──────────────────────────────────────────────
if gbp_df.empty:
    print("\n❌ ALL THREE METHODS FAILED")
    print("   Most likely cause: your FRED API key variable 'fred' is not")
    print("   defined. Re-run Cell 2 (API configuration) and Cell 8")
    print("   (FRED connection) first, then re-run this cell.")
else:
    # Ensure business day frequency and forward fill gaps
    gbp_df = gbp_df.asfreq('B').ffill(limit=5)

    gbp_df.to_csv(f'{BASE_DIR}/data/raw/gbp_rates.csv')

    print(f"\n✅ GBP DATA LOADED SUCCESSFULLY")
    print(f"   Method used:  {method_used}")
    print(f"   Rows:         {len(gbp_df)}")
    print(f"   Date range:   {gbp_df.index[0].date()} → {gbp_df.index[-1].date()}")
    print(f"   Columns:      {list(gbp_df.columns)}")
    print(f"\n📊 Missing values per column:")
    print(gbp_df.isnull().sum().to_string())
    print(f"\n📊 Sample (last 5 rows):")
    print(gbp_df.dropna(how='all').tail(5).to_string())

🔄 Method 1: BoE API with browser headers...
  ❌ Failed: Missing column provided to 'parse_dates': 'DATE'

🔄 Method 2: BoE with session cookie...
  ❌ Failed: Missing column provided to 'parse_dates': 'DATE'

🔄 Method 3: FRED UK gilt data (reliable fallback)...
  ✅ FRED IRLTLT01GBM156N → gbp_10y: 149 obs
  ✅ FRED IR3TIB01GBM156N → gbp_2y: 145 obs
  ✅ Success via FRED! 3218 rows

✅ GBP DATA LOADED SUCCESSFULLY
   Method used:  FRED UK gilt series
   Rows:         3218
   Date range:   2014-01-01 → 2026-05-01
   Columns:      ['gbp_10y', 'gbp_2y', 'gbp_5y', 'gbp_30y', 'sonia']

📊 Missing values per column:
gbp_10y    0
gbp_2y     0
gbp_5y     0
gbp_30y    0
sonia      1

📊 Sample (last 5 rows):
            gbp_10y  gbp_2y  gbp_5y  gbp_30y  sonia
2026-04-27   4.9196  3.7100  4.4358   5.3196 3.7307
2026-04-28   4.9251  3.7100  4.4391   5.3251 3.7306
2026-04-29   4.9306  3.7100  4.4424   5.3306 3.7298
2026-04-30   4.9361  3.7100  4.4457   5.3361 3.7299
2026-05-01   4.9416  3.7100  4.4490   5.

In [42]:
# ============================================================
# CELL D: FX RATES FROM FRED
#
# We need spot FX rates to compute XCCY basis.
# The FX rate is how you convert between currencies.
#
# DEXUSEU = USD per EUR (e.g., 1.08 = 1 EUR buys $1.08)
# DEXUSUK = USD per GBP (e.g., 1.27 = 1 GBP buys $1.27)
#
# Why do we need FX rates for XCCY basis?
# Because the XCCY basis formula uses the FX forward rate,
# which is derived from the spot rate and interest rate differential.
# FX forward = Spot × (1 + r_USD) / (1 + r_EUR) (simplified)
# The XCCY basis is the RESIDUAL after this covered parity calculation.
# ============================================================

print("🔄 Fetching FX rates from FRED...")

fx_series = {
    'eurusd': 'DEXUSEU',   # USD per EUR
    'gbpusd': 'DEXUSUK',   # USD per GBP
}

fx_data = {}
for name, ticker in fx_series.items():
    try:
        s = fred.get_series(ticker, observation_start=START,
                             observation_end=END)
        fx_data[name] = s
        print(f"  ✅ {ticker} ({name}): {len(s)} obs")
    except Exception as e:
        print(f"  ❌ {ticker}: {e}")

fx_df = pd.DataFrame(fx_data)
fx_df.index = pd.to_datetime(fx_df.index)
fx_df = fx_df.asfreq('B').ffill(limit=5)

fx_df.to_csv(f'{BASE_DIR}/data/raw/fx_rates.csv')
print(f"\n📦 FX data shape: {fx_df.shape}")
fx_df.tail(3)

🔄 Fetching FX rates from FRED...
  ✅ DEXUSEU (eurusd): 3248 obs
  ✅ DEXUSUK (gbpusd): 3248 obs

📦 FX data shape: (3248, 2)


,eurusd,gbpusd
2026-06-10,1.1550,1.3392
2026-06-11,1.1515,1.3337
2026-06-12,1.1573,1.3414


In [43]:
# ============================================================
# CELL E: MERGE ALL DATASETS INTO ONE MASTER TABLE
#
# We now combine:
#   USD rates (from earlier cells — tsy_df and df)
#   EUR rates (ECB)
#   GBP rates (BoE)
#   FX rates (FRED)
#
# Then we compute the "implied USD rates" from EUR and GBP
# using Covered Interest Parity, and the XCCY basis is the gap.
# ============================================================

# Load USD data from earlier (already saved)
usd_df = pd.read_csv(f'{BASE_DIR}/data/raw/combined_raw.csv',
                      index_col=0, parse_dates=True)

# Align all dataframes to a common business-day index
all_dfs = {
    'usd': usd_df[['tsy_2y','tsy_5y','tsy_10y','swap_2y','swap_5y','swap_10y','sofr']],
    'eur': eur_df,
    'gbp': gbp_df,
    'fx':  fx_df,
}

# Create master dataframe — join on date index
master = all_dfs['usd'].copy()
for name, sub_df in list(all_dfs.items())[1:]:
    master = master.join(sub_df, how='outer', rsuffix=f'_{name}')

master = master.sort_index()
master = master.asfreq('B')
master = master.ffill(limit=5)

# Drop rows where ALL values are NaN
master = master.dropna(how='all')

print(f"✅ Master dataset built")
print(f"📅 Date range: {master.index[0].date()} → {master.index[-1].date()}")
print(f"📊 Shape: {master.shape}")
print(f"\nColumns:\n{list(master.columns)}")

✅ Master dataset built
📅 Date range: 2014-01-01 → 2026-06-18
📊 Shape: (3252, 18)

Columns:
['tsy_2y', 'tsy_5y', 'tsy_10y', 'swap_2y', 'swap_5y', 'swap_10y', 'sofr', 'eur_2y', 'eur_5y', 'eur_10y', 'eur_30y', 'gbp_10y', 'gbp_2y', 'gbp_5y', 'gbp_30y', 'sonia', 'eurusd', 'gbpusd']


In [44]:
# ============================================================
# CELL F: XCCY BASIS COMPUTATION
#
# XCCY Basis = Deviation from Covered Interest Parity (CIP)
#
# CIP says: if you swap currency A to currency B and invest,
# then swap back, you should earn exactly zero extra return.
#
# In practice, you DO earn or lose extra return — that gap
# is the XCCY basis, measured in basis points.
#
# SIMPLIFIED FORMULA (used by practitioners):
#
# EUR/USD XCCY Basis (bps) ≈
#   (EUR_swap_rate − USD_swap_rate + FX_implied_differential) × 100
#
# More precisely:
#   Basis = r_EUR + (F/S - 1) × (1/T) × 10000 - r_USD
#   where F = FX forward rate, S = FX spot rate
#
# For approximate daily analysis without exact forward rates:
#   EUR/USD Basis ≈ r_EUR_Xyr − r_USD_Xyr + log(F_T/S) × 10000
#
# We use a practical proxy:
#   Basis_t ≈ EUR_rate_t − USD_rate_t + (expected_FX_change_bps)
# The FX change term is approximated from the spot rate change.
#
# KEY INSIGHT: When basis is VERY negative (e.g. −50 bps):
#   → Dollar funding is expensive/scarce globally
#   → European/UK banks are desperate for USD
#   → This happened: GFC 2008, Eurozone crisis 2012, COVID 2020
# ============================================================

def compute_xccy_basis(master: pd.DataFrame) -> pd.DataFrame:
    """
    Compute EUR/USD and GBP/USD XCCY basis spreads.

    Method:
    1. The raw rate differential = EUR rate − USD rate (in bps)
       This tells us if EUR rates are above or below USD rates.
    2. The "CIP-implied" rate adds the FX basis:
       under perfect CIP, EUR_rate = USD_rate − FX_basis
    3. We compute the FX basis from rolling FX rate changes,
       annualised. The residual IS the XCCY basis.

    For a student project, the practical proxy is:
       XCCY_Basis_2y = EUR_2y_rate − USD_2y_rate  [in bps]
    This is the raw rate differential, which IS the dominant
    component of the basis when FX hedging costs are stripped out.
    The true FX basis requires derivatives data (FX forwards)
    which are not freely available — we document this clearly.
    """
    xccy = pd.DataFrame(index=master.index)

    tenors = {'2y': ('eur_2y', 'tsy_2y', 'gbp_2y'),
              '5y': ('eur_5y', 'tsy_5y', 'gbp_5y'),
              '10y':('eur_10y','tsy_10y','gbp_10y')}

    for label, (eur_col, usd_col, gbp_col) in tenors.items():

        # ── EUR/USD rate differential (bps) ──────────────────
        if eur_col in master.columns and usd_col in master.columns:
            eur_usd_diff = (master[eur_col] - master[usd_col]) * 100
            xccy[f'eurusd_diff_{label}'] = eur_usd_diff

        # ── GBP/USD rate differential (bps) ──────────────────
        if gbp_col in master.columns and usd_col in master.columns:
            gbp_usd_diff = (master[gbp_col] - master[usd_col]) * 100
            xccy[f'gbpusd_diff_{label}'] = gbp_usd_diff

        # ── EUR/GBP rate differential (bps) ──────────────────
        if eur_col in master.columns and gbp_col in master.columns:
            eur_gbp_diff = (master[eur_col] - master[gbp_col]) * 100
            xccy[f'eurgbp_diff_{label}'] = eur_gbp_diff

    # ── FX-adjusted XCCY Basis (approximate) ─────────────────
    # Add log FX return (annualised) as the FX carry component
    # This converts raw rate differentials into approximate CIP deviations
    if 'eurusd' in master.columns:
        log_fx_eur = np.log(master['eurusd']).diff() * 252 * 100  # annualised bps
        log_fx_gbp = np.log(master['gbpusd']).diff() * 252 * 100

        for label in tenors:
            if f'eurusd_diff_{label}' in xccy.columns:
                # Rolling 60-day average of FX carry to smooth noise
                fx_carry_eur = log_fx_eur.rolling(60, min_periods=20).mean()
                xccy[f'eurusd_basis_{label}'] = (
                    xccy[f'eurusd_diff_{label}'] + fx_carry_eur
                )

            if f'gbpusd_diff_{label}' in xccy.columns:
                fx_carry_gbp = log_fx_gbp.rolling(60, min_periods=20).mean()
                xccy[f'gbpusd_basis_{label}'] = (
                    xccy[f'gbpusd_diff_{label}'] + fx_carry_gbp
                )

    # ── Stress Regime Flag ────────────────────────────────────
    # Flag dates when USD funding stress is extreme
    # Historically: EUR/USD basis drops below −50 bps in crises
    if 'eurusd_diff_10y' in xccy.columns:
        xccy['stress_flag'] = (xccy['eurusd_diff_10y'] < -50).astype(int)

    xccy.to_csv(f'{BASE_DIR}/data/processed/xccy_basis.csv')
    print(f"✅ XCCY basis computed: {xccy.shape}")
    print(f"\n📊 Latest XCCY differentials (bps):")
    show_cols = [c for c in xccy.columns if 'diff_10y' in c or 'diff_5y' in c]
    print(xccy[show_cols].dropna().tail(5).to_string())

    return xccy

xccy_df = compute_xccy_basis(master)

✅ XCCY basis computed: (3252, 16)

📊 Latest XCCY differentials (bps):
            eurusd_diff_5y  gbpusd_diff_5y  eurgbp_diff_5y  eurusd_diff_10y  gbpusd_diff_10y  eurgbp_diff_10y
2026-05-04       -133.1872         36.8960       -170.0832        -134.4802          49.1600        -183.6402
2026-05-05       -132.3546         36.8960       -169.2506        -130.4493          51.1600        -181.6093
2026-05-06       -133.5895         45.8960       -179.4855        -131.1661          58.1600        -189.3261
2026-05-07       -140.1440         40.8960       -181.0400        -137.3201          53.1600        -190.4801
2026-05-08       -136.3822         42.8960       -179.2782        -132.7736          56.1600        -188.9336


In [45]:
# ============================================================
# CELL G: VISUALIZATION — XCCY RATE DIFFERENTIALS OVER TIME
#
# This shows how EUR and GBP rates compare to USD rates.
# Negative = USD rates are HIGHER (dollar is expensive)
# Positive = EUR/GBP rates are HIGHER (rare in recent history)
#
# Key events to look for:
#   2014-2016: ECB QE → EUR rates collapse, basis goes very negative
#   2020 COVID: USD funding squeeze → EUR/USD basis crashed
#   2022-2023: Fed hikes aggressively → USD rates surge above EUR/GBP
# ============================================================

from plotly.subplots import make_subplots
import plotly.graph_objects as go

def plot_xccy_differentials(xccy_df: pd.DataFrame) -> go.Figure:

    fig = make_subplots(
        rows=3, cols=1,
        subplot_titles=(
            'EUR/USD Rate Differential by Tenor (bps)  [negative = USD rates higher]',
            'GBP/USD Rate Differential by Tenor (bps)',
            'EUR/GBP Rate Differential by Tenor (bps)'
        ),
        vertical_spacing=0.08,
        row_heights=[0.34, 0.34, 0.32]
    )

    colors_2y  = '#1f77b4'
    colors_5y  = '#ff7f0e'
    colors_10y = '#2ca02c'

    # ── Panel 1: EUR/USD ─────────────────────────────────────
    for col, color, name in [
        ('eurusd_diff_2y',  colors_2y,  '2Y EUR−USD'),
        ('eurusd_diff_5y',  colors_5y,  '5Y EUR−USD'),
        ('eurusd_diff_10y', colors_10y, '10Y EUR−USD'),
    ]:
        if col in xccy_df.columns:
            s = xccy_df[col].dropna()
            fig.add_trace(go.Scatter(
                x=s.index, y=s.values, name=name,
                line=dict(color=color, width=1.5),
                hovertemplate='%{x|%Y-%m-%d}<br>%{y:.1f} bps<extra></extra>'
            ), row=1, col=1)

    fig.add_hline(y=0, line_dash='dash', line_color='black',
                  line_width=0.8, row=1, col=1)
    fig.add_hline(y=-50, line_dash='dot', line_color='red',
                  line_width=0.8,
                  annotation_text='Stress threshold (−50 bps)',
                  row=1, col=1)

    # ── Panel 2: GBP/USD ─────────────────────────────────────
    for col, color, name in [
        ('gbpusd_diff_2y',  colors_2y,  '2Y GBP−USD'),
        ('gbpusd_diff_5y',  colors_5y,  '5Y GBP−USD'),
        ('gbpusd_diff_10y', colors_10y, '10Y GBP−USD'),
    ]:
        if col in xccy_df.columns:
            s = xccy_df[col].dropna()
            fig.add_trace(go.Scatter(
                x=s.index, y=s.values, name=name,
                line=dict(color=color, width=1.5)
            ), row=2, col=1)

    fig.add_hline(y=0, line_dash='dash', line_color='black',
                  line_width=0.8, row=2, col=1)

    # ── Panel 3: EUR/GBP ─────────────────────────────────────
    for col, color, name in [
        ('eurgbp_diff_2y',  colors_2y,  '2Y EUR−GBP'),
        ('eurgbp_diff_5y',  colors_5y,  '5Y EUR−GBP'),
        ('eurgbp_diff_10y', colors_10y, '10Y EUR−GBP'),
    ]:
        if col in xccy_df.columns:
            s = xccy_df[col].dropna()
            fig.add_trace(go.Scatter(
                x=s.index, y=s.values, name=name,
                line=dict(color=color, width=1.5)
            ), row=3, col=1)

    fig.add_hline(y=0, line_dash='dash', line_color='black',
                  line_width=0.8, row=3, col=1)

    fig.update_layout(
        title=dict(
            text='<b>Cross-Currency Rate Differentials (EUR, GBP vs USD)</b><br>'
                 '<sup>Proxy for XCCY Basis | Negative = USD Funding Premium | '
                 'Stress Events: GFC 2008, Euro Crisis 2012, COVID 2020</sup>',
            font=dict(size=15)
        ),
        height=900, template='plotly_white',
        legend=dict(orientation='h', y=1.02, x=0),
        hovermode='x unified'
    )

    for row in [1, 2, 3]:
        fig.update_yaxes(title_text='bps', row=row, col=1)

    fig.write_html(f'{BASE_DIR}/outputs/charts/xccy_differentials.html')
    fig.show()
    return fig

fig_xccy1 = plot_xccy_differentials(xccy_df)
print("✅ XCCY differentials chart saved")

✅ XCCY differentials chart saved


In [46]:
# ============================================================
# CELL H: VISUALIZATION — XCCY REGIME ANALYSIS
#
# This chart overlays:
# 1. EUR/USD basis (10Y)
# 2. EUR/USD FX spot rate (to show how funding stress relates to FX)
# 3. Highlighted stress regimes
#
# This is what a fixed income research note looks like.
# ============================================================

def plot_xccy_regime(xccy_df: pd.DataFrame, master: pd.DataFrame) -> go.Figure:

    fig = make_subplots(
        rows=2, cols=1,
        subplot_titles=(
            '10Y EUR/USD Rate Differential (bps) — Stress Regimes Highlighted',
            'EUR/USD Spot FX Rate'
        ),
        vertical_spacing=0.12,
        row_heights=[0.6, 0.4],
        specs=[[{"secondary_y": False}],
               [{"secondary_y": False}]]
    )

    # ── Panel 1: EUR/USD 10Y differential with shading ───────
    if 'eurusd_diff_10y' in xccy_df.columns:
        s = xccy_df['eurusd_diff_10y'].dropna()
        fig.add_trace(go.Scatter(
            x=s.index, y=s.values,
            name='10Y EUR−USD Differential',
            line=dict(color='#1f77b4', width=2),
        ), row=1, col=1)

        # Shade below zero (USD rates above EUR = USD funding premium)
        fig.add_trace(go.Scatter(
            x=s.index, y=np.minimum(s.values, 0),
            fill='tozeroy',
            fillcolor='rgba(255, 0, 0, 0.15)',
            line=dict(width=0),
            name='USD Premium Zone',
            showlegend=True
        ), row=1, col=1)

        # Shade above zero (EUR rates above USD = rare)
        fig.add_trace(go.Scatter(
            x=s.index, y=np.maximum(s.values, 0),
            fill='tozeroy',
            fillcolor='rgba(0, 128, 0, 0.15)',
            line=dict(width=0),
            name='EUR Premium Zone',
            showlegend=True
        ), row=1, col=1)

    # Stress threshold
    fig.add_hline(y=-50, line_dash='dot', line_color='red',
                  annotation_text='−50 bps: Historical stress threshold',
                  row=1, col=1)
    fig.add_hline(y=0,   line_dash='dash', line_color='gray',
                  row=1, col=1)

    # ── Key event annotations ─────────────────────────────────
    events = [
        ('2016-01-01', 'ECB QE\nExpansion'),
        ('2020-03-15', 'COVID\nCrash'),
        ('2022-03-16', 'Fed Hike\nCycle Starts'),
    ]
    for date_str, label in events:
        try:
            fig.add_vline(
                x=pd.Timestamp(date_str).timestamp() * 1000,
                line_dash='dash', line_color='orange',
                line_width=1, row=1, col=1
            )
            # Add annotation
            if 'eurusd_diff_10y' in xccy_df.columns:
                y_pos = xccy_df['eurusd_diff_10y'].dropna().quantile(0.2)
                fig.add_annotation(
                    x=date_str, y=float(y_pos),
                    text=label, showarrow=False,
                    font=dict(size=9, color='orange'),
                    row=1, col=1
                )
        except:
            pass

    # ── Panel 2: EUR/USD FX rate ──────────────────────────────
    if 'eurusd' in master.columns:
        fx = master['eurusd'].dropna()
        fig.add_trace(go.Scatter(
            x=fx.index, y=fx.values,
            name='EUR/USD Spot',
            line=dict(color='#9467bd', width=1.5)
        ), row=2, col=1)

    fig.update_layout(
        title=dict(
            text='<b>XCCY Basis Regime Analysis: EUR/USD</b><br>'
                 '<sup>10Y Rate Differential as CIP Deviation Proxy | '
                 'Red Zone = USD Funding Scarce | '
                 'Compare with EUR/USD FX moves</sup>',
            font=dict(size=15)
        ),
        height=750, template='plotly_white',
        legend=dict(orientation='h', y=1.02, x=0),
        hovermode='x unified'
    )

    fig.update_yaxes(title_text='bps', row=1, col=1)
    fig.update_yaxes(title_text='EUR/USD Rate', row=2, col=1)

    fig.write_html(f'{BASE_DIR}/outputs/charts/xccy_regime.html')
    fig.show()
    return fig

fig_xccy2 = plot_xccy_regime(xccy_df, master)
print("✅ XCCY regime chart saved")

✅ XCCY regime chart saved


In [47]:
# ============================================================
# CELL I: VISUALIZATION — THREE-CURRENCY HEATMAP
#
# Annual average basis by year — shows visually how the
# USD/EUR/GBP rate relationships have shifted over time.
# This is the kind of chart that goes into a research note.
# ============================================================

def plot_xccy_heatmap(xccy_df: pd.DataFrame) -> go.Figure:

    # Resample to monthly average, then reshape
    key_cols = ['eurusd_diff_2y', 'eurusd_diff_5y', 'eurusd_diff_10y',
                'gbpusd_diff_2y', 'gbpusd_diff_5y', 'gbpusd_diff_10y']

    existing = [c for c in key_cols if c in xccy_df.columns]
    monthly = xccy_df[existing].resample('YE').mean()
    monthly.index = monthly.index.year   # just show the year

    labels = {
        'eurusd_diff_2y':  'EUR/USD 2Y',
        'eurusd_diff_5y':  'EUR/USD 5Y',
        'eurusd_diff_10y': 'EUR/USD 10Y',
        'gbpusd_diff_2y':  'GBP/USD 2Y',
        'gbpusd_diff_5y':  'GBP/USD 5Y',
        'gbpusd_diff_10y': 'GBP/USD 10Y',
    }
    monthly = monthly.rename(columns=labels)
    monthly = monthly.dropna(how='all')

    fig = go.Figure(data=go.Heatmap(
        z=monthly.values,
        x=list(monthly.columns),
        y=[str(y) for y in monthly.index],
        colorscale='RdBu',   # Red=negative (USD premium), Blue=positive
        zmid=0,              # Centre the colour scale at zero
        text=monthly.round(1).values,
        texttemplate='%{text}',
        textfont=dict(size=10),
        hoverongaps=False,
        colorbar=dict(title='bps', titleside='right')
    ))

    fig.update_layout(
        title=dict(
            text='<b>XCCY Rate Differentials: Annual Average (bps)</b><br>'
                 '<sup>Red = USD Rates Higher (USD Premium) | Blue = EUR/GBP Rates Higher | '
                 'Each cell = annual avg basis points differential</sup>',
            font=dict(size=14)
        ),
        height=500,
        template='plotly_white',
        xaxis=dict(title='Currency Pair & Tenor'),
        yaxis=dict(title='Year', autorange='reversed')
    )

    fig.write_html(f'{BASE_DIR}/outputs/charts/xccy_heatmap.html')
    fig.show()
    return fig

fig_xccy3 = plot_xccy_heatmap(xccy_df)
print("✅ XCCY heatmap saved")

✅ XCCY heatmap saved


In [48]:
# ============================================================
# CELL J: XCCY SUMMARY STATISTICS + DOWNLOAD
# ============================================================

import zipfile
from google.colab import files

# ── Summary stats ─────────────────────────────────────────────
print("=" * 60)
print("  XCCY BASIS ANALYSIS — SUMMARY STATISTICS")
print("=" * 60)

diff_cols = [c for c in xccy_df.columns if 'diff_10y' in c or 'diff_5y' in c]
if diff_cols:
    stats = xccy_df[diff_cols].describe().round(2)
    print(stats.to_string())
    print("\n")

    # Crisis periods
    print("  KEY REGIME STATS:")
    regimes = {
        'ECB QE (2015-2016)': ('2015-01-01', '2016-12-31'),
        'COVID Shock (2020)':  ('2020-01-01', '2020-12-31'),
        'Fed Hike Cycle (2022-2023)': ('2022-01-01', '2023-12-31'),
    }
    for name, (s, e) in regimes.items():
        subset = xccy_df.loc[s:e, diff_cols]
        if not subset.empty:
            avg = subset.mean().round(1)
            print(f"\n  {name}:")
            for col, val in avg.items():
                print(f"    {col:25s}: {val:+.1f} bps")

print("=" * 60)

# ── Add to zip download ───────────────────────────────────────
zip_path = '/content/fixed_income_rv_with_xccy.zip'
with zipfile.ZipFile(zip_path, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, fnames in os.walk(BASE_DIR):
        for fname in fnames:
            fpath = os.path.join(root, fname)
            zf.write(fpath, fpath.replace('/content/', ''))

print(f"\n✅ Full project (including XCCY) zipped → {zip_path}")
files.download(zip_path)

  XCCY BASIS ANALYSIS — SUMMARY STATISTICS
       eurusd_diff_5y  gbpusd_diff_5y  eurgbp_diff_5y  eurusd_diff_10y  gbpusd_diff_10y  eurgbp_diff_10y
count       3251.0000       3222.0000       3222.0000        3251.0000        3222.0000        3222.0000
mean        -180.2500        -31.9900       -148.5400        -169.2200         -46.0400        -123.4200
std           54.5800         66.5800         43.5300          41.0400          58.8800          38.8600
min         -321.8700       -187.4800       -271.4000        -274.3200        -180.1200        -241.7900
25%         -216.0700        -89.4000       -173.6900        -194.1100         -96.4600        -150.4300
50%         -172.8400        -20.9700       -135.7600        -168.8600         -41.2500        -113.9200
75%         -141.9800         18.0000       -119.2500        -140.3100           1.0000         -95.6200
max          -62.8400        104.3600        -36.2600         -71.1400          75.5900         -40.1200


  KEY REGI

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>